In [ ]:
import pandas as pd

orders = pd.read_csv("/content/orders_raw.csv")
products = pd.read_csv("/content/products.csv")

print("Orders:", orders.shape)
print("Products:", products.shape)

display(orders.head())
display(products.head())

Orders: (508, 11)
Products: (31, 5)


,order_id,order_date,customer_name,city,category,product_id,quantity,amount_inr,payment_mode,status,rating
0,1,2026-03-09,Dev,Pune,Snacks & Beverages,13,5,175.0,UPI,Delivered,5.0
1,2,2026-05-22,Ayaan,Bengaluru,Snacks & Beverages,15,1,60.0,Debit Card,Delivered,5.0
2,3,2026-06-30,Yash,Bengaluru,Household Essentials,25,2,150.0,UPI,Delivered,2.0
3,4,2026-03-01,Sneha,Hyderabad,Snacks & Beverages,11,1,30.0,UPI,Delivered,4.0
4,5,2026-04-05,Rohit,Mumbai,Snacks & Beverages,14,2,220.0,Debit Card,Delivered,3.0


,product_id,product_name,category,supplier,unit_price_inr
0,1,Banana 1kg,Fruits & Vegetables,FreshFarms Co,50
1,2,Tomato 1kg,Fruits & Vegetables,FreshFarms Co,40
2,3,Onion 1kg,Fruits & Vegetables,GreenValley Traders,35
3,4,Apple 1kg,Fruits & Vegetables,GreenValley Traders,180
4,5,Spinach Bunch,Fruits & Vegetables,FreshFarms Co,25


In [ ]:
# Verify 3 outlier rows after clipping
print("Upper fence:", upper_fence)

print(df.loc[outlier_indices, ["order_id", "amount_inr"]].head(3))

In [ ]:
df = orders.merge(products, on="product_id", how="left")

df["order_date"] = pd.to_datetime(df["order_date"])
df["month"] = df["order_date"].dt.to_period("M").astype(str)

monthly_category_revenue = (
    df.groupby(["category_x", "month"])
      .agg(
          order_count=("order_id", "count"),
          total_revenue=("amount_inr", "sum")
      )
      .reset_index()
)

monthly_category_revenue["avg_revenue"] = (
    monthly_category_revenue["total_revenue"]
    / monthly_category_revenue["order_count"]
)

display(monthly_category_revenue.head(10))

,category_x,month,order_count,total_revenue,avg_revenue
0,Bakery,2026-05,1,195.0,195.0
1,Bakery,2026-06,1,135.0,135.0
2,Dairy & Eggs,2026-02,1,225.0,225.0
3,Dairy & Eggs,2026-03,2,225.0,112.5
4,Fruits & Vegetables,2026-03,1,720.0,720.0
5,Household Essentials,2026-05,1,300.0,300.0
6,Personal Care,2026-06,1,40.0,40.0
7,BAKERY,2026-01,1,85.0,85.0
8,BAKERY,2026-03,1,350.0,350.0
9,BAKERY,2026-04,1,225.0,225.0


In [10]:
# Task 10 verification: IQR clipping on a copy

import pandas as pd

# Work on a copy so the original data is not changed
verify_df = orders.copy()

Q1 = verify_df["amount_inr"].quantile(0.25)
Q3 = verify_df["amount_inr"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

# Identify rows that were above the upper fence BEFORE clipping
outlier_indices = verify_df.index[
    verify_df["amount_inr"] > upper_fence
]

print("Upper fence:", upper_fence)
print("Number of outliers:", len(outlier_indices))

# Re-run clipping on the copy
verify_df["amount_inr"] = verify_df["amount_inr"].clip(upper=upper_fence)

# Manually check 3 previously-outlier rows
print("\nThree verified rows after clipping:")
print(verify_df.loc[
    outlier_indices[:3],
    ["order_id", "amount_inr"]
])

print("\nAll three equal upper fence:",
      (verify_df.loc[outlier_indices[:3], "amount_inr"] == upper_fence).all())

Upper fence: 574.375
Number of outliers: 23

Three verified rows after clipping:
     order_id  amount_inr
76         77     574.375
78         79     574.375
120       121     574.375

All three equal upper fence: True


In [ ]:
# Clean column name
monthly_category_revenue = monthly_category_revenue.rename(
    columns={"category_x": "category"}
)

# Save final report
monthly_category_revenue.to_csv(
    "/content/monthly_category_revenue.csv",
    index=False
)

print("File saved successfully!")
display(monthly_category_revenue.head(10))

File saved successfully!


,category,month,order_count,total_revenue,avg_revenue
0,Bakery,2026-05,1,195.0,195.0
1,Bakery,2026-06,1,135.0,135.0
2,Dairy & Eggs,2026-02,1,225.0,225.0
3,Dairy & Eggs,2026-03,2,225.0,112.5
4,Fruits & Vegetables,2026-03,1,720.0,720.0
5,Household Essentials,2026-05,1,300.0,300.0
6,Personal Care,2026-06,1,40.0,40.0
7,BAKERY,2026-01,1,85.0,85.0
8,BAKERY,2026-03,1,350.0,350.0
9,BAKERY,2026-04,1,225.0,225.0
